In [ ]:
import pandas as pd
import os
import re

DATA_DIR = os.path.dirname(os.path.abspath('mystocks.ipynb'))

accounts = {}  # account_number (str) -> DataFrame(index=Symbol, cols=[Quantity, Current Price])
acc = ['618',"370",'472',"137","133"]

def clean_num(val):
    """Strip $, commas, % from a value and return float or NaN."""
    if pd.isna(val):
        return float('nan')
    s = str(val).strip().replace('$', '').replace(',', '').replace('%', '')
    try:
        return float(s)
    except ValueError:
        return float('nan')

exclude = ["earnings.csv","historical.csv", "portfolio.csv",
           "sectors.csv","file_clean.csv","History_for_Account_226998197.csv",
            "History_for_Account_236369828.csv" ]
for filename in sorted(os.listdir(DATA_DIR)):
    if filename in exclude:
        continue
    filepath = os.path.join(DATA_DIR, filename)
    if not filename.endswith('.csv'):
        if not filename.endswith('.xlsx'):
            continue
        else:
            try:
                cols = pd.read_excel(filename, nrows=1, skiprows=11,index_col=0)
                cols = cols.iloc[0].to_list()
                df = pd.read_excel(filename,header=12, sheet_name=0, index_col=0, names =cols)
                acct_num = str(int(df['Account Number'].dropna().iloc[0][-4:]))
                #
                df['Quantity']      = pd.to_numeric(df['Quantity'], errors='coerce')
                df['Current_Price'] = df['Price'].apply(clean_num)
                df['Market_Value'] = df['Market Value'].apply(clean_num)
                result = result = (
                    df[['Security ID', 'Quantity', 'Current_Price','Market_Value']]
              .dropna(subset=['Market_Value'])
                      .set_index('Security ID').rename(index={'PGC': 'cash'})
                )
                accounts[acct_num] = result
    
            except Exception as e:
                print("no excel:",e)
                continue
            continue

    
 
    with open(filepath, 'r') as f:
        first_line = f.readline().strip()

    # ── Format A: Schwab/TD  "Positions for account ..." ──────────────────────
    if first_line.startswith('"Positions for account'):
        match = re.search(r'\.\.(\w+)', first_line)
        
        acct_num = match.group(1) if match else filename
        print(acct_num)

        df = pd.read_csv(filepath, skiprows=2, header=0)
        # Drop the spurious empty trailing column produced by the trailing comma
        df = df.loc[:, df.columns.notna() & (df.columns.str.strip() != '')]
        df.columns = df.columns.str.strip()

        NON_EQUITY = {
            #'Cash & Cash Investments', 
            'Futures Cash',
            'Futures Positions Market Value', 'Positions Total', '--'
        }
        df = df[df['Symbol'].notna() & ~df['Symbol'].isin(NON_EQUITY)]
        
        df['Quantity']      = pd.to_numeric(
            df['Qty (Quantity)'].astype(str).str.replace(',', '', regex=False),
            errors='coerce'
        )
        df['Current_Price'] = (
            df['Mkt Val (Market Value)']
              .apply(clean_num)
            / df['Quantity']
        )
        df['Market_Value'] = (
            df['Mkt Val (Market Value)']
              .apply(clean_num)
            
        )
        result = (
            df[['Symbol', 'Quantity',"Market_Value", 'Current_Price']]
              .dropna(subset=[ 'Market_Value'])
              .set_index('Symbol').rename(index={'Cash & Cash Investments': 'cash'})
        )
        accounts[acct_num] = result
        df.rename(index={'Cash & Cash Investments':"cash"}, inplace=True)

    # ── Format B: ETRADE PortfolioDownload  (Account Summary header) ────────
    elif first_line.strip('"') == 'Account Summary':
        with open(filepath, 'r') as f:
            raw = f.readlines()

        # Account number is the last numeric segment on line 3 (index 2)
        match = re.search(r'-(\d+)', raw[2])
        acct_num = match.group(1) if match else filename
        print(acct_num)
        # Locate the data-header row that begins with "Symbol,Last Price"
        header_idx = next(
            (i for i, line in enumerate(raw)
             if line.strip().startswith('Symbol,Last Price')),
            None
        )
        if header_idx is None:
            print(f'WARNING: could not find data header in {filename}')
            continue
        
        try:
            df = pd.read_csv(filepath, skiprows=header_idx, header=0, skipfooter=5,engine='python')
        except Exception as e:
            print(e)
            with open(filepath, 'r') as f:
                lines = f.readlines()

            lines = [line.rstrip(',\n') + '\n' for line in lines]

            with open('file_clean.csv', 'w') as f:
                f.writelines(lines)
            filepath = os.path.join(DATA_DIR, "file_clean.csv")
            df = pd.read_csv(filepath, skiprows=header_idx, header=0, skipfooter=5,engine='python')
        df.columns = df.columns.str.strip()
        
        df = df[df['Symbol'].notna() & ~df['Symbol'].isin([ 'TOTAL', ''])]

        df['Quantity']      = pd.to_numeric(df['Quantity'], errors='coerce')
        df['Current_Price'] = pd.to_numeric(df['Last Price $'], errors='coerce')
        df['Market_Value'] = pd.to_numeric(df['Value $'], errors='coerce')

        result = (
            df[['Symbol', 'Quantity', 'Current_Price','Market_Value']]
              .dropna(subset=['Market_Value'])
              .set_index('Symbol').rename(index={'CASH': 'cash'})
        )
        accounts[acct_num] = result

    # ── Format C: Fidelity Portfolio_Positions  (Account Number column) ───────
    else:
        try:
            with open(filepath, 'r') as f:
                lines = f.readlines()

            lines = [line.rstrip(',\n') + '\n' for line in lines]

            with open('file_clean.csv', 'w') as f:
                f.writelines(lines)
            filepath = os.path.join(DATA_DIR, "file_clean.csv")
            df = pd.read_csv(filepath, header=0)
        except Exception as e:
            print(f'WARNING: could not read {filename}: {e}')
            continue

        if 'Account Number' not in df.columns:
            print(f'WARNING: unrecognised format in {filename}, skipping')
            continue

        df.columns = df.columns.str.strip()
        
        acct_num = str(int(df['Account Number'].dropna().iloc[0][-4:]))

        

        df['Quantity']      = pd.to_numeric(df['Quantity'], errors='coerce')
        df['Current_Price'] = df['Last Price'].apply(clean_num)
        df['Market_Value'] = df['Current Value'].apply(clean_num)
        result = (
            df[['Symbol', 'Quantity', 'Current_Price','Market_Value']]
              .dropna(subset=['Market_Value'])
              .set_index('Symbol').rename(index={'SPAXX**': 'cash'})
        )
        accounts[acct_num] = result
    


# ── Print all accounts sorted by account number ───────────────────────────────
for acct in sorted(accounts):
    print(f"\n{'='*55}")
    print(f" Account: {acct}")
    print(f"{'='*55}")
    print(accounts[acct].to_string(float_format='%.4f'))


133
137
472
4919
Expected 10 fields in line 27, saw 11
1297
Expected 10 fields in line 18, saw 11
370
618

 Account: 1297
        Quantity  Current_Price  Market_Value
Symbol                                       
EWP      40.0000        56.6100     2264.4000
QUAL     20.0000       207.1550     4143.1000
SPYV     10.0000        59.8800      598.8000
SPY      23.0000       720.6500    16574.9500
ROST      5.0000       228.8400     1144.2000
QQQ       8.0000       674.1500     5393.2000
cash         NaN            NaN      557.0700

 Account: 133
        Quantity  Market_Value  Current_Price
Symbol                                       
BRK/B     6.0000     2838.0600       473.0100
cash         NaN      265.1800            NaN

 Account: 137
        Quantity  Market_Value  Current_Price
Symbol                                       
AAPL    106.0000    29694.8400       280.1400
AMZN    120.0000    32191.2000       268.2600
AXP       5.0000     1598.4000       319.6800
BRK/B    18.0000    

In [13]:
# Merge all accounts into one DataFrame, grouped by Symbol
all_rows = pd.concat(accounts.values())
all_rows.index.name = 'Symbol'   # normalize across CSV and xlsx formats

cash_rows   = all_rows[all_rows.index == 'cash']
equity_rows = all_rows[all_rows.index != 'cash']

# Flag any symbols where price differs across accounts (tolerance: $0.01)
price_check = equity_rows.groupby('Symbol')['Current_Price'].agg(['min', 'max'])
price_mismatches = price_check[abs(price_check['max'] - price_check['min']) > 0.01]
if not price_mismatches.empty:
    print("⚠ Price mismatches across accounts (same symbol, different price):")
    print(price_mismatches.to_string())
    print()

# Equity: sum Quantity & Market_Value, mean Current_Price (consistent across accounts)
combined = (
    equity_rows
    .groupby('Symbol')
    .agg(
        Quantity      =('Quantity',      'sum'),
        Current_Price =('Current_Price', 'mean'),
        Market_Value  =('Market_Value',  'sum'),
    )
    .sort_index()
)

# Cash: sum Market_Value only (Quantity/Price are unreliable across formats)
total_cash = cash_rows['Market_Value'].sum()
cash_combined = pd.DataFrame(
    {'Quantity': [float('nan')], 'Current_Price': [1.0], 'Market_Value': [total_cash]},
    index=pd.Index(['cash'], name='Symbol')
)

combined = pd.concat([combined, cash_combined])
combined.rename(index={'BRK/B':"BRK-B"})
combined


,Quantity,Current_Price,Market_Value
Symbol,,,
AAPL,106.0,280.14,29694.84
AMKR,30.0,71.09,2132.70
AMZN,184.0,268.26,49359.84
AVGO,40.0,421.28,16851.20
AXP,10.0,319.68,3196.80
...,...,...,...
XLF,25.0,51.92,1298.00
XLP,30.0,84.17,2525.10
XOM,160.0,152.75,24440.00


In [14]:
import yfinance as yf
import numpy as np
from datetime import date, timedelta

HIST_FILE  = os.path.join(DATA_DIR, 'historical.csv')
HIST_YEARS = 10
today      = date.today()
start_full = pd.Timestamp(today - timedelta(days=365 * HIST_YEARS + 30))

# ── Load existing historical data (wide: Date index, symbol columns) ──────────
if os.path.exists(HIST_FILE):
    try:
        hist_df = pd.read_csv(HIST_FILE, index_col=0, parse_dates=True)
        hist_df.index.name = 'Date'
        hist_df.index = hist_df.index.tz_localize(None)
        print(f'Loaded {HIST_FILE}  ({len(hist_df)} rows, {len(hist_df.columns)} symbols)')
    except Exception as e:
        hist_df = pd.DataFrame()
        print(f'Could not load historical.csv ({e}) — fetching full 10yr history')
else:
    hist_df = pd.DataFrame()
    print('No historical.csv found — fetching full 10yr history')

symbols = combined.index[combined.index != 'cash'].tolist()
yf_sym  = lambda s: s.replace('/', '-')   # BRK/B -> BRK-B

# ── Fetch only the missing tail for each symbol ───────────────────────────────
updated = False
for sym in symbols:
    ysym = yf_sym(sym)
    if sym in hist_df.columns and not hist_df[sym].dropna().empty:
        last_date = hist_df[sym].dropna().index[-1].date()
        if last_date >= today:
            continue
        fetch_start = last_date + timedelta(days=1)
    else:
        fetch_start = start_full.date()
    try:
        raw = yf.Ticker(ysym).history(
            start=str(fetch_start),
            end=str(today + timedelta(days=1)),
            auto_adjust=True
        )
        if raw.empty:
            continue
        new_close = raw['Close'].rename(sym)
        new_close.index = new_close.index.tz_localize(None).normalize()
        if sym in hist_df.columns:
            hist_df[sym] = hist_df[sym].combine_first(new_close)
        else:
            hist_df = hist_df.reindex(hist_df.index.union(new_close.index))
            hist_df[sym] = new_close
        updated = True
    except Exception as e:
        print(f'WARNING (history) {sym}: {e}')

cutoff  = pd.Timestamp(today - timedelta(days=365 * HIST_YEARS))
hist_df = hist_df[hist_df.index >= cutoff].sort_index()
if updated:
    hist_df.to_csv(HIST_FILE)
    print(f'Saved {HIST_FILE}  ({len(hist_df)} rows, {len(hist_df.columns)} symbols)')

# ── Risk-free rate (13-week T-bill annualized) ────────────────────────────────
try:
    rf_annual = yf.Ticker('^IRX').history(period='5d')['Close'].iloc[-1] / 100
except Exception:
    rf_annual = 0.043
rf_daily = rf_annual / 252
print(f'Risk-free rate: {rf_annual:.2%}')

# ── Per-symbol metrics ────────────────────────────────────────────────────────
windows = {'1yr': 252, '6m': 126, '3m': 63}
rows = {}

for sym in symbols:
    ysym = yf_sym(sym)
    row  = {}

    # -- Live price, P/E, analyst targets --
    try:
        info = yf.Ticker(ysym).info
        # Price: currentPrice works for stocks; ETFs only expose regularMarketPrice
        live_price = info.get('currentPrice') or info.get('regularMarketPrice')
        row['Current_Price'] = live_price
        # Trailing P/E — use ratio directly; recalculate from EPS as fallback
        t_pe  = info.get('trailingPE')
        t_eps = info.get('trailingEps')
        if t_pe is None and live_price and t_eps:
            t_pe = live_price / t_eps
        row['Trailing_PE'] = round(t_pe, 2) if t_pe else float('nan')
        # Forward P/E — same fallback
        f_pe  = info.get('forwardPE')
        f_eps = info.get('forwardEps')
        if f_pe is None and live_price and f_eps:
            f_pe = live_price / f_eps
        row['Forward_PE'] = round(f_pe, 2) if f_pe else float('nan')
        # Analyst price targets
        row['Target_Mean']   = info.get('targetMeanPrice')
        row['Target_Median'] = info.get('targetMedianPrice')
        row['Target_High']   = info.get('targetHighPrice')
        row['Target_Low']    = info.get('targetLowPrice')
        row['Num_Analysts']  = info.get('numberOfAnalystOpinions')
    except Exception as e:
        print(f'WARNING (info) {sym}: {e}')
        live_price = None
        for k in ('Current_Price', 'Trailing_PE', 'Forward_PE',
                  'Target_Mean', 'Target_Median', 'Target_High',
                  'Target_Low', 'Num_Analysts'):
            row[k] = float('nan')

    # -- Historical vol, Sharpe, % gains --
    if sym in hist_df.columns:
        closes    = hist_df[sym].dropna()
        daily_ret = closes.pct_change().dropna()
        row['Ann_Vol'] = (
            round(daily_ret.std() * np.sqrt(252), 4)
            if len(daily_ret) > 20 else float('nan')
        )
        for label, days in windows.items():
            if len(daily_ret) < 20:
                row[f'Sharpe_{label}'] = float('nan')
                row[f'Gain_{label}']   = float('nan')
                continue
            subset = daily_ret.iloc[-days:]
            row[f'Sharpe_{label}'] = round(
                (subset.mean() - rf_daily) / subset.std() * np.sqrt(252), 3
            )
            n = min(days, len(closes) - 1)
            row[f'Gain_{label}'] = round(
                (closes.iloc[-1] / closes.iloc[-n] - 1) * 100, 2
            )
    else:
        row['Ann_Vol'] = float('nan')
        for label in windows:
            row[f'Sharpe_{label}'] = float('nan')
            row[f'Gain_{label}']   = float('nan')

    rows[sym] = row

metrics_df = pd.DataFrame.from_dict(rows, orient='index')
metrics_df.index.name = 'Symbol'

# ── Push live prices + Market_Value back into combined ────────────────────────
live_prices = metrics_df['Current_Price'].dropna()
combined.loc[live_prices.index, 'Current_Price'] = live_prices
combined.loc[combined.index != 'cash', 'Market_Value'] = (
    combined.loc[combined.index != 'cash', 'Quantity']
    * combined.loc[combined.index != 'cash', 'Current_Price']
)

# ── Join all other metrics ────────────────────────────────────────────────────
for col in metrics_df.columns:
    if col != 'Current_Price':
        combined[col] = metrics_df[col]

combined

Could not load historical.csv ('Date' is not in list) — fetching full 10yr history


$USD999997: possibly delisted; no timezone found


Saved /home/ai1/Desktop/fiData/historical.csv  (2514 rows, 93 symbols)
Risk-free rate: 3.60%


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: USD999997"}}}


,Quantity,Current_Price,Market_Value,Trailing_PE,Forward_PE,Target_Mean,Target_Median,Target_High,Target_Low,Num_Analysts,Ann_Vol,Sharpe_1yr,Gain_1yr,Sharpe_6m,Gain_6m,Sharpe_3m,Gain_3m
Symbol,,,,,,,,,,,,,,,,,
AAPL,106.0,287.44,30468.64,34.84,30.07,303.37620,310.000,355.00,215.0,42.0,0.2891,1.573,47.10,0.528,6.61,0.641,3.45
AMKR,30.0,72.27,2168.10,41.53,29.50,75.50000,75.000,90.00,60.0,8.0,0.5251,2.535,315.04,2.361,98.52,2.940,46.69
AMZN,184.0,271.17,49895.28,32.44,27.47,310.80950,315.000,370.00,207.0,62.0,0.3242,1.284,43.70,0.596,8.38,2.576,28.93
AVGO,40.0,412.56,16502.40,80.11,22.77,475.49295,477.500,630.00,360.0,42.0,0.3909,1.849,103.07,0.865,15.39,2.841,24.18
AXP,10.0,318.69,3186.90,19.88,15.84,361.57135,351.285,450.00,285.0,22.0,0.3180,0.586,16.58,-0.808,-12.41,-1.239,-10.98
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
XLF,25.0,51.55,1288.75,16.49,NaN,NaN,NaN,NaN,NaN,NaN,0.2217,0.257,5.86,-0.254,-1.15,-0.782,-4.51
XLP,30.0,83.98,2519.40,25.47,NaN,NaN,NaN,NaN,NaN,NaN,0.1472,0.247,5.92,1.465,12.10,-0.952,-3.97
XOM,160.0,146.58,23452.80,24.72,14.38,166.27274,165.000,195.00,123.0,22.0,0.2806,1.554,44.93,1.990,30.94,0.160,-1.00


In [16]:
PORT_FILE  = os.path.join(DATA_DIR, 'portfolio.csv')
combined.to_csv(PORT_FILE)

In [17]:
eq = combined[combined.index != 'cash'].copy()
N  = 10

def show(title, df, cols):
    print(f'\n{chr(8212)*60}')
    print(f'  {title}')
    print(f'{chr(8212)*60}')
    print(df[cols].to_string())

# Highest Trailing P/E
top_tpe = eq['Trailing_PE'].dropna().nlargest(N)
show('WARNING  Highest Trailing P/E (most expensive on earnings)',
     eq.loc[top_tpe.index], ['Trailing_PE', 'Forward_PE', 'Current_Price', 'Market_Value'])

# Highest Forward P/E
top_fpe = eq['Forward_PE'].dropna().nlargest(N)
show('WARNING  Highest Forward P/E (market expects slow earnings growth)',
     eq.loc[top_fpe.index], ['Forward_PE', 'Trailing_PE', 'Current_Price', 'Market_Value'])

# Worst 3m Performance
worst_3m = eq['Gain_3m'].dropna().nsmallest(N)
show('WARNING  Worst 3-Month Performance',
     eq.loc[worst_3m.index], ['Gain_3m', 'Gain_6m', 'Gain_1yr', 'Sharpe_3m', 'Current_Price'])

# Lowest Forward P/E
low_fpe = eq['Forward_PE'].dropna().nsmallest(N)
show('OK  Lowest Forward P/E (potentially undervalued)',
     eq.loc[low_fpe.index], ['Forward_PE', 'Trailing_PE', 'Current_Price', 'Target_Mean', 'Market_Value'])

# Best 3m Performance
best_3m = eq['Gain_3m'].dropna().nlargest(N)
show('OK  Best 3-Month Performance',
     eq.loc[best_3m.index], ['Gain_3m', 'Gain_6m', 'Gain_1yr', 'Sharpe_3m', 'Current_Price'])



————————————————————————————————————————————————————————————
  WARNING  Highest Trailing P/E (most expensive on earnings)
————————————————————————————————————————————————————————————
        Trailing_PE  Forward_PE  Current_Price  Market_Value
Symbol                                                      
SHOP         109.55       48.16         111.74        223.48
AVGO          80.11       22.77         412.56      16502.40
SBUX          79.59       34.59         104.26      10426.00
TER           65.70       37.22         354.11       3541.10
COST          52.66       45.00        1012.06       2024.12
TXN           48.76       31.01         285.24       1426.20
WMT           47.69       39.62         130.20       7812.00
RDDT          46.84       18.60         163.95        163.95
PINS          44.90        9.62          21.55        215.50
CAT           44.54       30.56         895.69      13435.35

————————————————————————————————————————————————————————————
  WARNING  Highest For

In [18]:
# ── Sector / Cap-size / Vol-tier classification ───────────────────────────────
import yfinance as yf

SECTOR_CSV = os.path.join(DATA_DIR, 'sectors.csv')
eq = combined[combined.index != 'cash'].copy()
symbols = eq.index.tolist()
yf_sym  = lambda s: s.replace('/', '-')

# Map yfinance sector names → standard GICS labels
SECTOR_MAP = {
    'Technology':                  'Information Technology',
    'Financial Services':          'Financials',
    'Consumer Cyclical':           'Consumer Discretionary',
    'Consumer Defensive':          'Consumer Staples',
    'Basic Materials':             'Materials',
    'Communication Services':      'Communication Services',
    'Healthcare':                  'Healthcare',
    'Industrials':                 'Industrials',
    'Energy':                      'Energy',
    'Real Estate':                 'Real Estate',
    'Utilities':                   'Utilities',
}

# ETF category keyword → GICS sector (best-effort mapping)
ETF_SECTOR_KEYWORDS = {
    'Technology':         'Information Technology',
    'Financial':          'Financials',
    'Health':             'Healthcare',
    'Energy':             'Energy',
    'Real Estate':        'Real Estate',
    'Consumer':           'Consumer Discretionary',
    'Industrial':         'Industrials',
    'Utilities':          'Utilities',
    'Communication':      'Communication Services',
    'Materials':          'Materials',
    'Europe':             'International',
    'International':      'International',
    'Global':             'International',
    'Emerging':           'International',
    'Large Blend':        'Broad Market',
    'Large Growth':       'Broad Market',
    'Large Value':        'Broad Market',
    'Mid':                'Broad Market',
    'Small':              'Broad Market',
    'Multi-Asset':        'Broad Market',
    'Allocation':         'Broad Market',
    'Bond':               'Fixed Income',
    'High Yield':         'Fixed Income',
    'Gold':               'Commodities',
    'Commodity':          'Commodities',
}

# Cap-size boundaries (market cap in USD)
LARGE_CAP  = 10e9
MID_CAP    = 2e9

# Volatility tier thresholds (annualized vol)
HIGH_VOL   = 0.35
LOW_VOL    = 0.18

# Load cached sector data if it exists
if os.path.exists(SECTOR_CSV):
    sec_cache = pd.read_csv(SECTOR_CSV, index_col='Symbol').to_dict('index')
    print(f'Loaded sector cache: {len(sec_cache)} symbols')
else:
    sec_cache = {}

# Fetch info for any symbol not yet cached
needs_fetch = [s for s in symbols if s not in sec_cache]
for sym in needs_fetch:
    ysym = yf_sym(sym)
    try:
        info = yf.Ticker(ysym).info
        qtype = info.get('quoteType', '')
        if qtype == 'EQUITY':
            raw_sector = info.get('sector', '')
            sector     = SECTOR_MAP.get(raw_sector, raw_sector or 'Unknown')
            mktcap     = info.get('marketCap')
        else:
            # ETF / MUTUALFUND: derive sector from category
            cat    = info.get('category', '') or ''
            sector = next(
                (v for k, v in ETF_SECTOR_KEYWORDS.items() if k.lower() in cat.lower()),
                'Broad Market'
            )
            mktcap = None   # ETF market cap is fund AUM, not useful for cap-size

        sec_cache[sym] = {
            'Quote_Type': qtype,
            'Sector':     sector,
            'MarketCap':  mktcap,
        }
    except Exception as e:
        print(f'WARNING (sector) {sym}: {e}')
        sec_cache[sym] = {'Quote_Type': 'Unknown', 'Sector': 'Unknown', 'MarketCap': None}

# Save updated cache
sec_df = pd.DataFrame.from_dict(sec_cache, orient='index')
sec_df.index.name = 'Symbol'
sec_df.to_csv(SECTOR_CSV)

# ── Attach to combined ────────────────────────────────────────────────────────
for col in ('Quote_Type', 'Sector', 'MarketCap'):
    combined.loc[combined.index != 'cash', col] = sec_df[col]

eq = combined[combined.index != 'cash'].copy()

# Cap-size tier (equities only; ETFs labelled 'ETF/Fund')
def cap_tier(row):
    if row['Quote_Type'] != 'EQUITY' or pd.isna(row['MarketCap']):
        return 'ETF/Fund'
    mc = row['MarketCap']
    if mc >= LARGE_CAP: return 'Large Cap'
    if mc >= MID_CAP:   return 'Mid Cap'
    return 'Small Cap'

# Vol tier
def vol_tier(row):
    v = row.get('Ann_Vol')
    if pd.isna(v): return 'Unknown'
    if v >= HIGH_VOL:  return 'High Vol'
    if v <= LOW_VOL:   return 'Low Vol'
    return 'Mid Vol'

eq['Cap_Tier'] = eq.apply(cap_tier, axis=1)
eq['Vol_Tier'] = eq.apply(vol_tier, axis=1)

# ── Aggregation helper ────────────────────────────────────────────────────────
GAIN_COLS = ['Gain_3m', 'Gain_6m', 'Gain_1yr']

def sector_summary(df, group_col):
    g = df.groupby(group_col)
    mv  = g['Market_Value'].sum().rename('Total_Market_Value')
    # Weighted-average gains (weight = market value of each position)
    wgains = {}
    for gc in GAIN_COLS:
        sub = df[['Market_Value', gc, group_col]].dropna(subset=[gc])
        wgains[gc] = (
            sub.groupby(group_col)
               .apply(lambda x: (x[gc] * x['Market_Value']).sum() / x['Market_Value'].sum(),
                      include_groups=False)
        )
    gain_df = pd.DataFrame(wgains).round(2)
    result  = pd.concat([mv, gain_df], axis=1).sort_values('Total_Market_Value', ascending=False)
    result['Total_Market_Value'] = result['Total_Market_Value'].map('${:,.0f}'.format)
    return result

# ── Print all three groupings ─────────────────────────────────────────────────
def print_summary(title, df):
    print(f'\n{"═"*65}')
    print(f'  {title}')
    print(f'{"═"*65}')
    print(df.to_string())

print_summary('By GICS Sector',       sector_summary(eq, 'Sector'))
print_summary('By Market-Cap Tier',   sector_summary(eq, 'Cap_Tier'))
print_summary('By Volatility Tier',   sector_summary(eq, 'Vol_Tier'))

Loaded sector cache: 114 symbols

═════════════════════════════════════════════════════════════════
  By GICS Sector
═════════════════════════════════════════════════════════════════
                       Total_Market_Value  Gain_3m  Gain_6m  Gain_1yr
Sector                                                               
Broad Market                     $224,728     8.73    14.03     43.84
Information Technology           $125,112    32.84    54.13    193.56
Consumer Discretionary            $71,079    20.58    11.11     38.48
Communication Services            $46,690    10.20    13.97     72.28
Financials                        $43,837    -4.32     3.44     22.93
Energy                            $29,794     5.60    37.87     51.95
Industrials                       $23,588    10.80    34.60    119.17
Consumer Staples                  $11,780     3.21    20.85     13.47
International                      $9,906     0.55     9.23     21.93
Real Estate                        $1,569     2

In [19]:
# ── Analyst Target Analysis ───────────────────────────────────────────────────
tgt = combined[combined.index != 'cash'].copy()
tgt = tgt[tgt['Target_Median'].notna() & tgt['Current_Price'].notna()]

# Upside/downside ratio: Target_Median / Current_Price
tgt['Target_Upside'] = ((tgt['Target_Median'] / tgt['Current_Price'] - 1) * 100).round(2)

# Target spread as % of median: (High - Low) / Median
tgt['Target_Spread'] = ((tgt['Target_High'] - tgt['Target_Low']) / tgt['Target_Median'] * 100).round(2)

def show(title, df, cols):
    print(f'\n{chr(8213)*62}')
    print(f'  {title}')
    print(f'{chr(8213)*62}')
    print(df[cols].to_string())

# 1. Analyst median target BELOW current price
overvalued = tgt[tgt['Target_Median'] < tgt['Current_Price']].sort_values('Target_Upside')
show(
    'WARNING  Analyst Median Target BELOW Current Price',
    overvalued,
    ['Current_Price', 'Target_Median', 'Target_Upside', 'Target_Low', 'Target_High', 'Num_Analysts']
)

# 2. Biggest upside to analyst median target (top 10)
most_upside = tgt.nlargest(10, 'Target_Upside')
show(
    'OK  Most Upside to Analyst Median Target (top 10)',
    most_upside,
    ['Current_Price', 'Target_Median', 'Target_Upside', 'Target_High', 'Num_Analysts']
)

# 3. Tightest analyst consensus (top 10 narrowest spread)
tightest = tgt.nsmallest(10, 'Target_Spread')
show(
    'OK  Tightest Analyst Consensus  (Target_High - Target_Low) / Target_Median  (top 10)',
    tightest,
    ['Current_Price', 'Target_Median', 'Target_Upside', 'Target_Spread', 'Num_Analysts']
)


――――――――――――――――――――――――――――――――――――――――――――――――――――――――――――――
  WARNING  Analyst Median Target BELOW Current Price
――――――――――――――――――――――――――――――――――――――――――――――――――――――――――――――
        Current_Price  Target_Median  Target_Upside  Target_Low  Target_High  Num_Analysts
Symbol                                                                                    
INTC           109.62      80.000000         -27.02   20.400000   118.000000          42.0
QCOM           202.55     160.000000         -21.01  100.000000   300.000000          30.0
MU             646.63     550.000000         -14.94  249.000000  1000.000000          42.0
BBVA            21.87      20.000000          -8.55   19.000000    28.560000           3.0
TXN            285.24     279.000000          -2.19  200.000000   340.000000          32.0
NVO             45.80      44.926277          -1.91   39.913685    65.859055          12.0
CBOE           338.65     335.000000          -1.08  273.000000   395.000000          14.0
C

In [ ]:
# ── Earnings dates, analyst recommendations, upgrades/downgrades ──────────────
from datetime import date, timedelta
import warnings
warnings.filterwarnings('ignore')

EARN_FILE  = os.path.join(DATA_DIR, 'earnings.csv')
today      = pd.Timestamp(date.today())

# ── Load or initialise earnings cache ────────────────────────────────────────
# Columns: Symbol, Next_Earnings, EPS_Est, Rev_Est_High, Rev_Est_Low
if os.path.exists(EARN_FILE):
    earn_cache = pd.read_csv(EARN_FILE, index_col='Symbol', parse_dates=['Next_Earnings'])
    print(f'Loaded earnings cache: {len(earn_cache)} symbols')
else:
    earn_cache = pd.DataFrame(columns=['Next_Earnings','EPS_Est','Rev_Est_High','Rev_Est_Low'])
    earn_cache.index.name = 'Symbol'
    print('No earnings.csv — fetching all')

symbols  = combined.index[combined.index != 'cash'].tolist()
yf_sym   = lambda s: s.replace('/', '-')
recs_rows = {}
upgrades  = []
etf_symbols = set(['POCT', 'QTOP', 'VGK', 'PPA', 'RSP', 'EUSA', 'JPXN', 'DRIV', 
              'QUAL', 'EAOR', 'DAX', 'EWY', 'QQQ', 'ECH', 'SPYG', 'IWM', 'EUFN',
                'EWP', 'FEZ', 'COLO', 'EFNL', 'EWW', 'EWS', 'IEV', 'IEUR', 'VUG', 
                'GDE', 'EEMV', 'EPOL', 'XLF', 'XLP', 'QVMT', 'EWI', 'SPVM', 'USD999997', 
                'SPYV', 'OPPJ', 'SPY', 'EWJ', 'RND'])
for sym in symbols:
    if sym in etf_symbols:
        continue
    ysym = yf_sym(sym)

    # -- Earnings: only re-fetch if not cached OR cached date has already passed --
    needs_earn = True
    if sym in earn_cache.index:
        cached_date = earn_cache.loc[sym, 'Next_Earnings']
        if pd.notna(cached_date) and pd.Timestamp(cached_date) > today:
            needs_earn = False   # still in the future — keep it

    if needs_earn:
        try:
            cal = yf.Ticker(ysym).calendar
            if cal and 'Earnings Date' in cal and cal['Earnings Date']:
                next_earn = pd.Timestamp(cal['Earnings Date'][0])
                earn_cache.loc[sym] = {
                    'Next_Earnings':  next_earn,
                    'EPS_Est':        cal.get('Earnings Average'),
                    'Rev_Est_High':   cal.get('Revenue High'),
                    'Rev_Est_Low':    cal.get('Revenue Low'),
                }
            else:
                earn_cache.loc[sym, 'Next_Earnings'] = pd.NaT
        except Exception as e:
            earn_cache.loc[sym, 'Next_Earnings'] = pd.NaT

    # -- Recommendations summary (current month) --
    try:
        rs = yf.Ticker(ysym).recommendations_summary
        if rs is not None and not rs.empty:
            cur = rs[rs['period'] == '0m'].iloc[0]
            total = cur[['strongBuy','buy','hold','sell','strongSell']].sum()
            recs_rows[sym] = {
                'Strong_Buy':  int(cur['strongBuy']),
                'Buy':         int(cur['buy']),
                'Hold':        int(cur['hold']),
                'Sell':        int(cur['sell']),
                'Strong_Sell': int(cur['strongSell']),
                'Consensus':   (
                    'Strong Buy'  if cur['strongBuy'] / max(total,1) > 0.4 else
                    'Buy'         if (cur['strongBuy']+cur['buy']) / max(total,1) > 0.5 else
                    'Hold'        if cur['hold'] / max(total,1) > 0.4 else
                    'Sell'        if (cur['sell']+cur['strongSell']) / max(total,1) > 0.4 else
                    'Mixed'
                )
            }
    except Exception:
        pass

    # -- Upgrades/downgrades: last 90 days --
    try:
        ud = yf.Ticker(ysym).upgrades_downgrades
        if ud is not None and not ud.empty:
            cutoff = pd.Timestamp(today - timedelta(days=90), tz='UTC')
            ud.index = pd.to_datetime(ud.index, utc=True)
            recent = ud[ud.index >= cutoff].copy()
            if not recent.empty:
                recent.insert(0, 'Symbol', sym)
                upgrades.append(recent.reset_index()[['Symbol','GradeDate','Firm','ToGrade','FromGrade','Action','currentPriceTarget']])
    except Exception:
        pass

# Save earnings cache
earn_cache.to_csv(EARN_FILE)
print(f'Saved earnings cache: {EARN_FILE}')

# ── Attach recs to combined ───────────────────────────────────────────────────
recs_df = pd.DataFrame.from_dict(recs_rows, orient='index')
recs_df.index.name = 'Symbol'
for col in recs_df.columns:
    combined[col] = recs_df[col]



Loaded earnings cache: 94 symbols
Saved earnings cache: /home/ai1/Desktop/fiData/earnings.csv


In [26]:
# ── Export all tables to JSON for the viewer app ─────────────────────────────
import json as _json
from datetime import date

APP_DATA = os.path.join(DATA_DIR, 'app_data')
os.makedirs(APP_DATA, exist_ok=True)

def _df_to_records(df):
    """Convert a DataFrame to a JSON-serialisable list of dicts."""
    return _json.loads(df.reset_index().to_json(orient='records', date_format='iso'))

# 1. Per-account tables
acct_out = {}
for acct_id, df in accounts.items():
    acct_out[acct_id] = _df_to_records(df)
with open(os.path.join(APP_DATA, 'accounts.json'), 'w') as f:
    _json.dump(acct_out, f, indent=2)

# 2. Combined portfolio
with open(os.path.join(APP_DATA, 'combined.json'), 'w') as f:
    _json.dump(_df_to_records(combined), f, indent=2)

# ── Sector summaries (rebuild compactly from combined) ────────────────────────
eq = combined[combined.index != 'cash'].copy()
GAIN_COLS = ['Gain_3m', 'Gain_6m', 'Gain_1yr']

def _sector_summary(df, group_col):
    if group_col not in df.columns:
        return {}
    g   = df.groupby(group_col)
    mv  = g['Market_Value'].sum().rename('Total_Market_Value')
    wg  = {}
    for gc in GAIN_COLS:
        sub = df[['Market_Value', gc, group_col]].dropna(subset=[gc])
        wg[gc] = sub.groupby(group_col).apply(
            lambda x: (x[gc] * x['Market_Value']).sum() / x['Market_Value'].sum(),
            include_groups=False
        ).round(2)
    result = pd.concat([mv, pd.DataFrame(wg)], axis=1)
    result['Total_Market_Value'] = result['Total_Market_Value'].round(2)
    return _df_to_records(result.reset_index())

sector_data = {
    'by_gics':    _sector_summary(eq, 'Sector'),
    'by_cap':     _sector_summary(eq, 'Cap_Tier'),
    'by_vol':     _sector_summary(eq, 'Vol_Tier'),
}
with open(os.path.join(APP_DATA, 'sectors.json'), 'w') as f:
    _json.dump(sector_data, f, indent=2)

# ── Flags ─────────────────────────────────────────────────────────────────────
N_FLAG = 10
flags  = {}
flag_cols = {
    'high_trailing_pe': ('Trailing_PE', 'largest'),
    'high_forward_pe':  ('Forward_PE',  'largest'),
    'worst_3m':         ('Gain_3m',     'smallest'),
    'low_forward_pe':   ('Forward_PE',  'smallest'),
    'best_3m':          ('Gain_3m',     'largest'),
}
for key, (col, direction) in flag_cols.items():
    if col not in eq.columns:
        continue
    sub = eq[col].dropna()
    idx = sub.nlargest(N_FLAG).index if direction == 'largest' else sub.nsmallest(N_FLAG).index
    keep = [c for c in ['Trailing_PE','Forward_PE','Current_Price','Market_Value',
                         'Gain_3m','Gain_6m','Gain_1yr','Sharpe_3m','Target_Mean'] if c in eq.columns]
    flags[key] = _df_to_records(eq.loc[idx, keep])
with open(os.path.join(APP_DATA, 'flags.json'), 'w') as f:
    _json.dump(flags, f, indent=2)

# ── Analyst targets ───────────────────────────────────────────────────────────
tgt_cols = [c for c in ['Current_Price','Target_Median','Target_High','Target_Low',
                          'Target_Mean','Target_Upside','Target_Spread','Num_Analysts'] if c in eq.columns]
if 'Target_Median' in eq.columns and 'Current_Price' in eq.columns:
    tgt = eq[tgt_cols].dropna(subset=['Target_Median','Current_Price']).copy()
    if 'Target_Upside' not in tgt.columns:
        tgt['Target_Upside'] = ((tgt['Target_Median'] / tgt['Current_Price'] - 1) * 100).round(2)
    if 'Target_Spread' not in tgt.columns and 'Target_High' in tgt.columns:
        tgt['Target_Spread'] = ((tgt['Target_High'] - tgt['Target_Low']) / tgt['Target_Median'] * 100).round(2)
    targets_data = {
        'overvalued':   _df_to_records(tgt[tgt['Target_Median'] < tgt['Current_Price']].sort_values('Target_Upside')),
        'most_upside':  _df_to_records(tgt.nlargest(10, 'Target_Upside')),
        'tightest':     _df_to_records(tgt.nsmallest(10, 'Target_Spread')) if 'Target_Spread' in tgt.columns else [],
    }
else:
    targets_data = {}
with open(os.path.join(APP_DATA, 'targets.json'), 'w') as f:
    _json.dump(targets_data, f, indent=2)

# ── Earnings ──────────────────────────────────────────────────────────────────
EARN_FILE = os.path.join(DATA_DIR, 'earnings.csv')
if os.path.exists(EARN_FILE):
    earn_df = pd.read_csv(EARN_FILE, index_col='Symbol', parse_dates=['Next_Earnings'])
    today   = pd.Timestamp(date.today())
    upcoming = (
        earn_df[earn_df.index.isin(combined.index) &
                earn_df['Next_Earnings'].notna() &
                (earn_df['Next_Earnings'] > today)]
        .sort_values('Next_Earnings')
        .reset_index()
    )
    with open(os.path.join(APP_DATA, 'earnings.json'), 'w') as f:
        _json.dump(_json.loads(upcoming.to_json(orient='records', date_format='iso')), f, indent=2)

# ── Recommendations ───────────────────────────────────────────────────────────
rec_cols = [c for c in ['Strong_Buy','Buy','Hold','Sell','Strong_Sell','Consensus'] if c in combined.columns]
if rec_cols:
    recs_out = combined[rec_cols].dropna(how='all')
    with open(os.path.join(APP_DATA, 'recommendations.json'), 'w') as f:
        _json.dump(_df_to_records(recs_out), f, indent=2)

# ── Upgrades / Downgrades ────────────────────────────────────────────────────
UD_FILE = os.path.join(DATA_DIR, 'upgrades.csv')
if os.path.exists(UD_FILE):
    ud_df = pd.read_csv(UD_FILE)
    with open(os.path.join(APP_DATA, 'upgrades.json'), 'w') as f:
        _json.dump(_json.loads(ud_df.to_json(orient='records', date_format='iso')), f, indent=2)

print(f"Saved app data to {APP_DATA}/")
for fn in sorted(os.listdir(APP_DATA)):
    path = os.path.join(APP_DATA, fn)
    print(f"  {fn}  ({os.path.getsize(path):,} bytes)")

Saved app data to /home/ai1/Desktop/fiData/app_data/
  accounts.json  (16,227 bytes)
  combined.json  (63,025 bytes)
  earnings.json  (8,131 bytes)
  flags.json  (14,155 bytes)
  recommendations.json  (8,207 bytes)
  sectors.json  (2,130 bytes)
  targets.json  (7,612 bytes)


In [27]:
# ── Print: next earnings by date ──────────────────────────────────────────────
upcoming = (
    earn_cache[earn_cache['Next_Earnings'].notna() & (earn_cache['Next_Earnings'] > today)]
    .sort_values('Next_Earnings')
)
# Only show symbols we own
upcoming = upcoming[upcoming.index.isin(symbols)]

print(f'\n{"═"*62}')
print('  Upcoming Earnings (soonest first)')
print(f'{"═"*62}')
print(upcoming[['Next_Earnings','EPS_Est','Rev_Est_High','Rev_Est_Low']].to_string())

# ── Print: buy/sell/hold summary ──────────────────────────────────────────────
if not recs_df.empty:
    print(f'\n{"═"*62}')
    print('  Analyst Recommendations (current month)')
    print(f'{"═"*62}')
    print(
        recs_df.sort_values(['Strong_Buy','Buy'], ascending=False)
               .to_string()
    )

# ── Print: recent upgrades/downgrades ────────────────────────────────────────
if upgrades:
    ud_all = pd.concat(upgrades).sort_values('GradeDate', ascending=False)
    print(f'\n{"═"*62}')
    print('  Upgrades / Downgrades  (last 90 days)')
    print(f'{"═"*62}')
    print(ud_all.to_string(index=False))



══════════════════════════════════════════════════════════════
  Upcoming Earnings (soonest first)
══════════════════════════════════════════════════════════════
       Next_Earnings   EPS_Est  Rev_Est_High   Rev_Est_Low
Symbol                                                    
PBR       2026-05-11       NaN  1.388191e+11  1.223885e+11
NVDA      2026-05-20   1.77461  8.551200e+10  7.789600e+10
ROST      2026-05-21   1.67575  5.645420e+09  5.496100e+09
WMT       2026-05-21   0.65875  1.768500e+11  1.722320e+11
AZO       2026-05-26  36.02555  4.912922e+09  4.789800e+09
COST      2026-05-28   4.94839  7.079800e+10  6.728400e+10
AVGO      2026-06-03   2.39061  2.239550e+10  2.187900e+10
MU        2026-06-24  18.97141  3.645800e+10  1.967600e+10
CCL       2026-06-24   0.33140  6.758711e+09  6.590000e+09
WFC       2026-07-14   1.70819  2.195100e+10  2.160100e+10
IBKR      2026-07-14   0.60656  1.718334e+09  1.660000e+09
GS        2026-07-14  13.72498  1.629400e+10  1.533200e+10
SCHW      2

In [11]:
earn_cache

,Next_Earnings,EPS_Est,Rev_Est_High,Rev_Est_Low
Symbol,,,,
AAPL,2026-07-30,1.89077,1.120000e+11,1.075010e+11
AMKR,2026-07-27,0.47177,1.822606e+09,1.773000e+09
AMZN,2026-07-30,1.81692,1.995570e+11,1.860000e+11
AVGO,2026-06-03,2.39061,2.239550e+10,2.187900e+10
AXP,2026-07-24,4.37935,1.970700e+10,1.945700e+10
...,...,...,...,...
WMT,2026-05-21,0.65875,1.768500e+11,1.722320e+11
XLF,NaT,NaN,NaN,NaN
XLP,NaT,NaN,NaN,NaN


In [ ]:

# ── Modern Portfolio Theory — Core Metrics ────────────────────────────────────
import matplotlib.pyplot as plt
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

MPT_LOOKBACK = 252  # ~1 trading year

# ── Returns matrix ────────────────────────────────────────────────────────────
eq_all = combined.index[combined.index != 'cash'].tolist()
avail  = [s for s in eq_all if s in hist_df.columns]

# Forward-fill prices to handle international ETF non-trading days
prices = hist_df[avail].iloc[-(MPT_LOOKBACK + 1):].ffill()
prices = prices.loc[:, prices.count() >= 200]   # drop symbols with sparse history
rets   = prices.pct_change().iloc[1:].fillna(0) # 0 return on non-trading days
mpt_syms = rets.columns.tolist()
n = len(mpt_syms)
print(f'{n} symbols in MPT universe  |  {len(rets)} trading days')

# ── Annualized mean & covariance ──────────────────────────────────────────────
mu  = rets.mean() * 252
cov = rets.cov()  * 252

# ── Current MV-weighted portfolio ─────────────────────────────────────────────
mv = combined.loc[mpt_syms, 'Market_Value'].fillna(0)
w0 = (mv / mv.sum()).values

def port_stats(w):
    r   = float(w @ mu.values)
    vol = float(np.sqrt(np.maximum(w @ cov.values @ w, 0)))
    sr  = (r - rf_annual) / vol if vol > 0 else 0.0
    return r, vol, sr

p_ret, p_vol, p_sr = port_stats(w0)

print(f'\n{"═"*62}')
print(f'  Current Portfolio  ({n} equity symbols, 1-yr lookback)')
print(f'{"═"*62}')
print(f'  Expected Return (ann):  {p_ret:>9.2%}')
print(f'  Volatility (ann):       {p_vol:>9.2%}')
print(f'  Sharpe Ratio:           {p_sr:>9.3f}')
print(f'  Risk-free Rate:         {rf_annual:>9.2%}')
print(f'  Total Equity MV:        ${mv.sum():>12,.0f}')

# ── Beta & Jensen's Alpha vs SPY ──────────────────────────────────────────────
if 'SPY' in rets.columns:
    spy_r = rets['SPY']
    spy_v = spy_r.var()
    ba = {}
    for sym in mpt_syms:
        b = rets[sym].cov(spy_r) / spy_v
        a = mu[sym] - b * (spy_r.mean() * 252)   # annualized Jensen's alpha
        ba[sym] = {'Beta': round(b, 3), 'Alpha_pct': round(a * 100, 2)}

    ba_df = pd.DataFrame(ba).T.astype(float)
    ba_df.index.name = 'Symbol'
    combined.loc[ba_df.index, 'Beta']      = ba_df['Beta']
    combined.loc[ba_df.index, 'Alpha_pct'] = ba_df['Alpha_pct']

    port_beta = float(w0 @ ba_df['Beta'].reindex(mpt_syms).fillna(1).values)
    print(f'  Portfolio Beta (vs SPY): {port_beta:>8.3f}')

    print(f'\n  High-beta positions (Beta > 1.5):')
    hb = ba_df[ba_df['Beta'] > 1.5].sort_values('Beta', ascending=False)
    print(hb.to_string() if not hb.empty else '  None')

    print(f"\n  Top 10 by Jensen's Alpha (ann %):")
    print(ba_df.nlargest(10, 'Alpha_pct').to_string())

# ── Risk Contribution ─────────────────────────────────────────────────────────
# RC_i = w_i * (Σw)_i / σ_p  — additive decomposition; sums to σ_p
Sw        = cov.values @ w0
rc_vol    = w0 * Sw / p_vol        # each asset's contribution to portfolio vol
rc_pct    = rc_vol / p_vol * 100   # as % of total vol — sums to 100

risk_df = pd.DataFrame({
    'Weight_pct':      (w0 * 100).round(2),
    'RiskContrib_pct': rc_pct.round(2),
}, index=mpt_syms)
risk_df.index.name = 'Symbol'

hhi   = ((w0 * 100) ** 2).sum()
eff_n = 1.0 / (w0 ** 2).sum()

print(f'\n{"═"*62}')
print(f'  Concentration — HHI: {hhi:.0f}/10000  |  Effective-N: {eff_n:.1f}')
print(f'{"═"*62}')
print(f'\n  Top 15 Risk Contributors (% of portfolio volatility):')
print(risk_df.nlargest(15, 'RiskContrib_pct').to_string())


In [ ]:

# ── Efficient Frontier (Monte Carlo + Optimization) ───────────────────────────
N_MC = 3000
rng  = np.random.default_rng(42)

mc_r, mc_v, mc_s = [], [], []
for _ in range(N_MC):
    w = rng.dirichlet(np.ones(n))
    r, v, s = port_stats(w)
    mc_r.append(r); mc_v.append(v); mc_s.append(s)

mc_r = np.array(mc_r) * 100
mc_v = np.array(mc_v) * 100
mc_s = np.array(mc_s)

# ── Max-Sharpe portfolio ──────────────────────────────────────────────────────
cons   = [{'type': 'eq', 'fun': lambda w: w.sum() - 1}]
bounds = [(0.0, 1.0)] * n
w_init = np.ones(n) / n

def neg_sharpe(w):
    r, v, _ = port_stats(w)
    return -(r - rf_annual) / v if v > 0 else 1e6

def min_vol_fn(w):
    _, v, _ = port_stats(w)
    return v

opt_ms = minimize(neg_sharpe,  w_init, method='SLSQP', bounds=bounds,
                  constraints=cons, options={'maxiter': 2000, 'ftol': 1e-9})
w_ms = opt_ms.x
r_ms, v_ms, s_ms = port_stats(w_ms)

# ── Min-Variance portfolio ────────────────────────────────────────────────────
opt_mv = minimize(min_vol_fn, w_init, method='SLSQP', bounds=bounds,
                  constraints=cons, options={'maxiter': 2000, 'ftol': 1e-9})
w_mv = opt_mv.x
r_mv, v_mv, s_mv = port_stats(w_mv)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 7))
sc = ax.scatter(mc_v, mc_r, c=mc_s, cmap='viridis', alpha=0.35, s=14, zorder=1)
plt.colorbar(sc, ax=ax, label='Sharpe Ratio')

ax.scatter(p_vol*100, p_ret*100, marker='*', s=420, color='red',
           zorder=5, label=f'Current  (SR={p_sr:.2f}, Vol={p_vol:.1%})')
ax.scatter(v_ms*100,  r_ms*100,  marker='D', s=150, color='gold',
           edgecolors='black', zorder=5,
           label=f'Max Sharpe  (SR={s_ms:.2f}, Vol={v_ms:.1%})')
ax.scatter(v_mv*100,  r_mv*100,  marker='s', s=150, color='cyan',
           edgecolors='black', zorder=5,
           label=f'Min Variance  (SR={s_mv:.2f}, Vol={v_mv:.1%})')

ax.set_xlabel('Annualized Volatility (%)', fontsize=12)
ax.set_ylabel('Annualized Expected Return (%)', fontsize=12)
ax.set_title(f'Efficient Frontier  (1-yr lookback, long-only, {n} assets)', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.25)
plt.tight_layout()

EF_PATH = os.path.join(DATA_DIR, 'efficient_frontier.png')
plt.savefig(EF_PATH, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved {EF_PATH}')

# ── Summary table ─────────────────────────────────────────────────────────────
print(f'\n{"═"*62}')
print(f'  Optimized vs Current')
print(f'{"═"*62}')
print(f'  {"Portfolio":<16} {"Return":>8} {"Vol":>8} {"Sharpe":>8}')
print(f'  {"-"*44}')
print(f'  {"Current":<16} {p_ret:>8.2%} {p_vol:>8.2%} {p_sr:>8.3f}')
print(f'  {"Max Sharpe":<16} {r_ms:>8.2%} {v_ms:>8.2%} {s_ms:>8.3f}')
print(f'  {"Min Variance":<16} {r_mv:>8.2%} {v_mv:>8.2%} {s_mv:>8.3f}')

print(f'\n  Max-Sharpe weights > 1%:')
ms_w = pd.Series(w_ms * 100, index=mpt_syms).round(2)
print(ms_w[ms_w > 1].sort_values(ascending=False).to_string())


In [ ]:

# ── Correlation Heatmap (top 25 positions by market value) ────────────────────
import seaborn as sns

TOP_N   = min(25, len(mpt_syms))
top_sym = combined.loc[mpt_syms, 'Market_Value'].nlargest(TOP_N).index.tolist()
corr    = rets[top_sym].corr().round(2)

fig, ax = plt.subplots(figsize=(15, 13))
sns.heatmap(
    corr, annot=True, fmt='.2f', cmap='RdYlGn_r',
    vmin=-1, vmax=1, center=0,
    linewidths=0.3, linecolor='gray',
    annot_kws={'size': 7}, ax=ax,
)
ax.set_title(
    f'Return Correlations — Top {TOP_N} Positions by Market Value\n(1-yr daily returns)',
    fontsize=13,
)
plt.tight_layout()

CORR_PATH = os.path.join(DATA_DIR, 'correlation_heatmap.png')
plt.savefig(CORR_PATH, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved {CORR_PATH}')

# ── Highly correlated pairs ────────────────────────────────────────────────────
THRESH = 0.85
high_corr = [
    (s1, s2, corr.loc[s1, s2])
    for i, s1 in enumerate(top_sym)
    for s2 in top_sym[i + 1:]
    if abs(corr.loc[s1, s2]) > THRESH
]
print(f'\n  Pairs with |correlation| > {THRESH}:')
if high_corr:
    for s1, s2, c in sorted(high_corr, key=lambda x: abs(x[2]), reverse=True):
        print(f'  {s1:8s} ↔ {s2:8s}  {c:+.2f}')
else:
    print('  None found above threshold')
